# 2026-10-05 Lab data 1: sessions, frames and the heading

**Author:** Kyle Shannon · **Notebook:** lab-data-1-sessions-frames-heading · **Created:** 2026-10-05T13:24:25-07:00

**Purpose:** How the lab's files become sessions and frames, what the tracking and the heading contain, how the light bar sits on the head (the heading offset), and how the head direction relates to the direction of travel.


In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.figure_format = "retina"

import subprocess
from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.ticker import MultipleLocator
import numpy as np
import pandas as pd
import scienceplots  # noqa: F401  registers the SciencePlots styles

import axis_of_travel_rnn
from environments import list_environments, load_environment
from axis_of_travel_rnn.store import resolve_store

plt.style.use(["science", "notebook", "no-latex"])
rng = np.random.default_rng(0)
store = resolve_store(None)
FIGURES = Path("figures")

commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True,
                        text=True).stdout.strip() or "unknown"
print(f"axis-of-travel-rnn {axis_of_travel_rnn.__version__}  commit {commit}  {date.today()}")
print(f"store: {store.root} ({store.source})")
print(f"environments: {', '.join(list_environments())}")

## Data

The cells load them through `src/proj_helpers/lab_data_access.py` (the reader, the catalog and a way to pick sessions) and nothing else. Every definition that encodes something we learned lives in the chapter that teaches it.

The plots are made from the lab's recordings, so run `pixi run nbstripout-install` once per clone: git then stores this notebook without its outputs.

`RUN_ALL = False` makes the loops that use `scope` cover a sample of 12 sessions, the first three of each group. `True` covers all 109, which takes minutes on the network volume. The totals quoted in the text are for all 109.


In [ ]:
from collections import Counter
from datetime import datetime

from proj_helpers.lab_data_access import NAME, catalog, find, lab_root, load_mat, pick_scope

RUN_ALL = True  # False: the loops over `scope` cover 3 sessions per group, for a quick look
root = lab_root()
sessions = catalog(root)  # session id -> {"group", "dvt" path, "analysis" path, "pattern"}
scope = pick_scope(sessions, run_all=RUN_ALL)
print(f"lab data: {root}")
print(f"{len(sessions)} sessions, the loops below cover {len(scope)}")


## Analysis

### 1.1 · From 220 files to 109 sessions

#### The question and the goal
Which files make a session, is the pairing always one tracking file plus one analysis file, and which names do
not parse? Goal: a session list built from names alone, with every odd name accounted for, and the vocabulary of
tags in the analysis file names (they carry the experimental condition).

#### What we found
- 220 `.mat` files: 218 parsed, 2 skipped by name: `NS14Data/NS14_rec27_02191500_tt_rmaps_bilatmusc.mat`
  (malformed stamp, a twin of `..._021915001_tt_rmaps_bilatmusc.mat`) and `allPLXFiles.mat` in the root.
- 109 sessions (JL1 14, NS14 32, NS15 33, NS16 30). Every one is exactly one ProcessedDVT file plus one paired analysis
  file.
- Analysis names, by variant tag: none 76 (`rmaps` or `tt_rmaps`); after `rmaps`: bilatACSF 4, bilatmusc 4,
  leftACSF 1, leftmusc 3 (all NS14), plus45 3, plusquarter or plusQuarter 7, plusquarterturn 6; before `rmaps`
  (all NS14): dark 1, plus45turn 2, plusquarterturn 2.

#### File Info
- A **session** is exactly one tracking file and one analysis file.
- Tags are matched without regard to case (`plusQuarter`) and may sit on either side of `rmaps`.


#### 1.1.1. The lab data root

In [ ]:
print("lab data root:", root)
for folder in sorted(p for p in root.iterdir() if p.is_dir()):
    print(f"{folder.name:15s} {len(list(folder.rglob('*.mat'))):4d} .mat files")
print("in the root itself:", [p.name for p in root.glob("*.mat")])

#### 1.1.2. Split each file name into its fields
The pattern is `<subject>_rec<N>_<MMDDYY><sss>[_tags]<kind>.mat`. Expect 218 parsed and 2 skipped. The skipped
twin has a two-digit session (`..._02191500_...`), which the pattern refuses.

In [ ]:
def parse(path):
    match = NAME.match(path.name)
    return None if match is None else {"group": path.parent.name, **match.groupdict(), "path": path}


parsed, skipped = [], []
for path in sorted(root.rglob("*.mat")):
    info = parse(path)
    if info is None:
        skipped.append(path)
    else:
        parsed.append(info)

print(f"{len(parsed)} parsed, {len(skipped)} skipped")
for path in skipped:
    print("  skipped:", path.relative_to(root))
for info in parsed[:3]:
    print({key: value for key, value in info.items() if key != "path"})


#### 1.1.3. Group the files into sessions
A session ID joins four fields from the file name: subject, recording number, date and session digits, for example `JL1_rec23_20150403_001`. The stamp in the name is month-day-year (`040315`). The ID uses year-month-day (ISO 8601), so it cannot be misread.

A file's role comes from the part of the name after the stamp:
- **tracking file**: exactly `ProcessedDVT`
- **analysis file**: `rmaps` is one of its `_`-separated tokens
- **other**: anything else


In [ ]:
def session_id(info):
    day = datetime.strptime(info["stamp"], "%m%d%y").strftime("%Y%m%d")
    return f"{info['subject']}_rec{info['rec']}_{day}_{info['session']}"


def role(info):
    tokens = [token.lower() for token in info["rest"].split("_")]
    if tokens == ["processeddvt"]:
        return "dvt"
    return "analysis" if "rmaps" in tokens else "other"


by_session = {}
for info in parsed:
    entry = by_session.setdefault(session_id(info), {"dvt": [], "analysis": [], "other": []})
    entry[role(info)].append(info["path"].name)

print(f"{len(by_session)} sessions")
print("files per session (tracking, analysis):", Counter((len(s["dvt"]), len(s["analysis"])) for s in by_session.values()))
print("files with no role:", sum(len(s["other"]) for s in by_session.values()))


#### 1.1.4. Sessions per group, and every shape an analysis file name takes

The tags are the experimental conditions. 

Note that the tag sits either after `rmaps` (`..._rmaps_bilatmusc`) or before it (`..._plus45turn_rmaps`), and that `plusquarter` also appears as `plusQuarter`.

In [ ]:
per_group = Counter(info["group"] for info in parsed if role(info) == "dvt")
print("sessions per group:", dict(per_group))

patterns = Counter((info["group"], info["rest"]) for info in parsed if role(info) == "analysis")
for (group, rest), n in sorted(patterns.items()):
    print(f"  {group:12s} {rest:28s} {n:3d}")

assert set(by_session) == set(sessions), "the notebook's grouping and the helper's catalog disagree"


#### Things to try later
- [ ] Which sessions share a recording number across days? (`rec<N>` is the recording, the date can repeat it.)
- [ ] Plot sessions against date for each rat. Are the infusion and rotation sessions clustered in time?
- [ ] Find the rows where the session digits are not `001`. What does `002` mean for that rat and day?

### 1.2 · Frames, LEDs and the Heading

#### Why this was examined
Everything the converter keeps comes from the tracking files, so every rule it checks them against had to be tested on the real files: frames are contiguous and 1/60 s apart, unfilled samples carry a flag and a sentinel position, the stored heading is the angle between the two lights, and positions stay inside the 640 by 480 camera frame. A converter built on untested rules would fail good sessions or pass bad ones.

#### The question and the goal
Do the frame rules hold across all 109 tracking files, in both MATLAB formats? Which columns of `processedDVT` do we use, which flags decide the NaNs, and what exactly is `HDRadians`? Goal: each rule confirmed or demoted, the columns settled, and an understanding of `head_dir` that the later RNN stages can respect.

#### What we found
- Frames run 1..N without gaps, and `rawDVT` and `processedDVT` agree on frame and time (hand-checked on the first session of each group).
- **The clock has pauses.** `t` is frame × 0.016667 s while the camera runs, but 32 of 109 files have at least one jump in `t`, where the recording stopped and restarted (from a few minutes up to about 56). The frame number does not jump. The lab's event times follow `t`, so they look shifted after a pause. Frames, not `t`, are the reliable clock.
- A lost sample sits at the sentinel position (1, 1) and carries a flag, and the two agree. Lost = filled + unfilled, never both. The heading is NaN exactly where either light is unfilled.
- The stored heading is `atan2(front − back)` from the two light positions, to 0.000 degrees over 69,347 rows. FLAG for the RNN stages: `head_dir` is a function of the two lights, so storing both lights stores the heading. Whatever the RNN is given must be one point, not both lights (AGENTS.md rule 1).
- The light bar is rigid: the lights are about 17 to 20 px apart in nearly every session. Heading jumps of more than 90 degrees between consecutive frames are rare, and nearly all come from frames where the lights are under 8 px apart: pixel jitter, not head turns.
- NS14_rec14 has 41 samples outside 640 by 480. Every one is a lab-filled sample, so the lab's gap filling overshot the frame.
- Heading reaches +pi, so a range check must be inclusive.
- Columns used: frame and time from `processedDVT` columns 0:2, front light 2:4, back light 4:6, flags `samplesFilled` and `samplesUnfilled`, heading `HDRadians`. Columns 6:10 are unused.

#### What it changed
- "Coordinates inside 640 by 480" went from a fail to a warn with a count. The values are kept as stored.
- The heading rules (NaN exactly when either light is unfilled, otherwise within plus or minus pi, inclusive) stay fail checks.
- `rawDVT` and `samplesLost` are no longer read by the converter.
- Frames are the clock. Each pause is recorded as a break in `t`, and anything that samples trajectories must not treat the two sides of a pause as continuous.

#### Where it lives
Spec: Behavior, the first two bullets. Dictionary: 1.1, 1.4 (Pauses) and 2.4.


#### 1.2.1. Size of all sessions' tracking and analysis files
The analysis files hold the neural data and are much larger than the tracking files. That is why the converter reads only the variables it names and never loads a whole analysis file.


In [ ]:
sizes = pd.DataFrame({
    "group": [session["group"] for session in scope.values()],
    "tracking MB": [session["dvt"].stat().st_size / 1e6 for session in scope.values()],
    "analysis MB": [session["analysis"].stat().st_size / 1e6 for session in scope.values()],
}, index=list(scope))

sizes_by_group = sizes.groupby("group").agg(
    sessions=("tracking MB", "size"),
    tracking_total=("tracking MB", "sum"), tracking_largest=("tracking MB", "max"),
    analysis_total=("analysis MB", "sum"), analysis_largest=("analysis MB", "max"),
).round(1)
print(sizes_by_group.to_string())
print(f"all {len(sizes)} sessions: tracking {sizes['tracking MB'].sum():.0f} MB, analysis {sizes['analysis MB'].sum():.0f} MB")

#### 1.2.2. Frame number and time
Each row of a tracking file is one camera frame, and the lab stores a time `t` for it. At 60 Hz, `t` should be the frame number times 0.016667 s. Two questions: does that hold in every file, and where it does not, what happened?

One row per session is drawn on the recording clock (the stored `t`, in minutes):
- **Pause (red)**: `t` steps by more than 0.1 s (6 frames) between two frames. The recording stopped and restarted, so the clock skips ahead while the frame number does not.
- **Arena, track, variant**: the segments the lab stamped, on the same clock.
- **Black bar**: both lights unfilled for more than 10 s.
- **Event times**: the lab's run events carry a row and a time. Each time is compared with the tracking `t` at its row, and with row × 0.016667.

Rows (frame numbers) stay the authority: frames are contiguous even when `t` jumps. The lab's event times follow the tracking `t`, so they look shifted wherever a pause came before them. Check the counts printed below against these claims: 32 of 109 files have a jump, and only JL1_rec35 and JL1_rec36 have event times off by a small amount (0.5 s and 0.3 s at their start) that no pause explains.


In [ ]:
DT = 0.016667                    # seconds per frame, as the lab stored it
MIN_JUMP_SECONDS = 0.1           # a step in the tracking time longer than this (6 frames) is a jump
LONG_GAP_FRAMES = 600            # both lights lost for longer than this (10 s) is a tracking gap
EVENT_TOLERANCE_SECONDS = 0.01
FRAMES_PER_MINUTE = 60 * 60
EPOCH_NAMES = ["arena", "track", "variant"]
EPOCH_COLORS = {"arena": "#f0c571", "track": "#59a89c", "variant": "#326b77"}   # gold, teal, and a darker teal
PAUSE_COLOR, LOST_COLOR, BOX_COLOR, GUIDE_COLOR = "#d47264", "#2b2b2b", "0.45", "0.85"


def locate_jump(frame_after_jump, epoch_spans):
    """Where a jump, between the frame before it and frame_after_jump, falls against the lab's epochs."""
    frame_before_jump = frame_after_jump - 1
    for epoch_name, (first_frame, last_frame) in zip(EPOCH_NAMES, epoch_spans):
        if first_frame <= frame_before_jump and frame_after_jump <= last_frame:
            return f"inside {epoch_name}"
    ended_before = [name for name, (_, last_frame) in zip(EPOCH_NAMES, epoch_spans) if last_frame < frame_after_jump]
    starts_after = [name for name, (first_frame, _) in zip(EPOCH_NAMES, epoch_spans) if first_frame > frame_before_jump]
    if ended_before and starts_after:
        return f"between {ended_before[-1]} and {starts_after[0]}"
    return "outside the epochs" if epoch_spans else "no timeStamps"


def long_runs(mask, longer_than):
    """Zero-based (first, last) index of each run of True values longer than longer_than."""
    edges = np.flatnonzero(np.diff(np.r_[0, mask.astype(int), 0]))
    return [(start, stop - 1) for start, stop in zip(edges[::2], edges[1::2]) if stop - start > longer_than]


def clip_spans(spans, containers):
    """The parts of each (start, end) span that lie inside one of the container spans."""
    return [(max(start, box_start), min(end, box_end)) for start, end in spans for box_start, box_end in containers
            if max(start, box_start) < min(end, box_end)]


def describe_tracking_clock(session):
    """One session's recording clock: jumps, epochs and lost stretches, and the event times against it."""
    analysis = load_mat(session["analysis"], variables=["spiral_events", "timeStamps"])
    tracking = load_mat(session["dvt"], variables=["processedDVT", "samplesUnfilled"])
    tracking_time = tracking["processedDVT"][:, 1]
    minutes = tracking_time / 60                                      # the recording clock, pauses included
    both_lights_lost = tracking["samplesUnfilled"].astype(bool).all(axis=1)
    steps = np.diff(tracking_time)
    jump_steps = np.flatnonzero(steps > MIN_JUMP_SECONDS)            # step i runs from frame i + 1 to frame i + 2
    stretch_edges = np.r_[0, jump_steps + 1, len(tracking_time)]     # zero-based index of each stretch's first frame
    epoch_spans = np.atleast_1d(analysis.get("timeStamps", [])).astype(int).reshape(-1, 2).tolist()
    recorded_stretches = [(minutes[first], minutes[stop - 1]) for first, stop in zip(stretch_edges[:-1], stretch_edges[1:])]
    lost_spans = [(minutes[first], minutes[last]) for first, last in long_runs(both_lights_lost, LONG_GAP_FRAMES)]

    events = np.asarray(analysis["spiral_events"], dtype=float).reshape(-1, 3)   # tracking row, time, label
    event_rows, event_times = events[:, 0].astype(int), events[:, 1]
    in_tracking = (event_rows >= 1) & (event_rows <= len(tracking_time))
    event_rows, event_times = event_rows[in_tracking], event_times[in_tracking]
    return {
        "group": session["group"],
        "frames": len(tracking_time),
        "epoch_spans": epoch_spans,                                  # frames, as the lab stamped them
        "jump_frames": (jump_steps + 2).tolist(),                    # the first frame after each jump
        "jump_seconds": steps[jump_steps].tolist(),
        "longest_stretch_minutes": np.diff(stretch_edges).max() / FRAMES_PER_MINUTE,
        "events_off_tracking_clock": int((np.abs(event_times - tracking_time[event_rows - 1]) > EVENT_TOLERANCE_SECONDS).sum()),
        "events_off_row_times_dt": int((np.abs(event_times - event_rows * DT) > EVENT_TOLERANCE_SECONDS).sum()),
        # the same things on the recording clock, in minutes, for the chart
        "recorded_stretches_min": recorded_stretches,
        "epochs_min": [(name, minutes[first - 1], minutes[last - 1]) for name, (first, last) in zip(EPOCH_NAMES, epoch_spans)],
        "pauses_min": [(minutes[step], minutes[step + 1]) for step in jump_steps],
        "lost_tracking_min": clip_spans(lost_spans, recorded_stretches),   # never drawn across a pause
    }


clock = {name: describe_tracking_clock(session) for name, session in scope.items()}

In [ ]:
per_session = pd.DataFrame.from_dict(clock, orient="index")
per_session["date"] = per_session.index.str.split("_").str[2]
per_session["recorded_minutes"] = per_session["frames"] / FRAMES_PER_MINUTE
per_session["jumps"] = per_session["jump_seconds"].str.len()
per_session["has_jump"] = per_session["jumps"] > 0
per_session["paused_minutes"] = per_session["jump_seconds"].map(sum) / 60

clock_by_group = per_session.groupby("group").agg(
    sessions=("frames", "size"),
    sessions_with_jumps=("has_jump", "sum"),
    jumps=("jumps", "sum"),
    paused_minutes=("paused_minutes", "sum"),
    median_recorded_minutes=("recorded_minutes", "median"),
    median_longest_stretch_minutes=("longest_stretch_minutes", "median"),
    events_off_tracking_clock=("events_off_tracking_clock", "sum"),
    events_off_row_times_dt=("events_off_row_times_dt", "sum"),
).round(1)

jump_table = pd.DataFrame([
    {"session": name, "frame_after_jump": frame, "seconds": seconds, "minutes": seconds / 60,
     "where": locate_jump(frame, summary["epoch_spans"])}
    for name, summary in clock.items()
    for frame, seconds in zip(summary["jump_frames"], summary["jump_seconds"])
])

print(clock_by_group.to_string())
print(f"\n{len(per_session)} tracking files; {per_session['has_jump'].sum()} have a jump in t, {len(jump_table)} jumps in all, "
      f"{jump_table['minutes'].min():.1f} to {jump_table['minutes'].max():.1f} min each")
print("where the jumps fall:", dict(jump_table["where"].value_counts()))
off_tracking_clock = per_session.index[per_session["events_off_tracking_clock"] > 0].tolist()
print(f"event times that differ from row * DT: {(per_session['events_off_row_times_dt'] > 0).sum()} sessions;  "
      f"from the tracking time at their row: {len(off_tracking_clock)} sessions {off_tracking_clock}")

In [ ]:
def spans_to_bars(spans):
    """(start, end) pairs as the (start, width) pairs that broken_barh draws."""
    return [(start, end - start) for start, end in spans]


def draw_rows(axis, names, per_session):
    """One row per session on the recording clock: a box per recorded stretch with its epochs, lost tracking, pauses."""
    for position, name in enumerate(names):
        summary = per_session.loc[name]
        axis.axhline(position, color=GUIDE_COLOR, linewidth=0.6, linestyle=":", zorder=0)   # hidden behind the boxes
        row_bar = (position - 0.4, 0.8)
        stretches = spans_to_bars(summary["recorded_stretches_min"])
        axis.broken_barh(stretches, row_bar, facecolor="white", zorder=1)
        for epoch_name, start, end in summary["epochs_min"]:
            axis.broken_barh([(start, end - start)], row_bar, facecolor=EPOCH_COLORS[epoch_name], zorder=2)
        for start, end in summary["pauses_min"]:
            axis.broken_barh([(start, end - start)], row_bar, facecolor=PAUSE_COLOR, zorder=2)
            if end - start >= 4:
                axis.text((start + end) / 2, position, f"{end - start:.0f}", ha="center", va="center",
                          fontsize=9, fontweight="bold", color="white", zorder=5)
        axis.broken_barh(spans_to_bars(summary["lost_tracking_min"]), (position + 0.05, 0.3), facecolor=LOST_COLOR, zorder=3)
        axis.broken_barh(stretches, row_bar, facecolor="none", edgecolor=BOX_COLOR, linewidth=0.8, zorder=4)   # perimeter last
    axis.set_yticks(range(len(names)))
    axis.set_yticklabels([" ".join(name.split("_")[1:3]) for name in names])   # recording number and date
    axis.set_ylim(len(names) - 0.5, -0.5)
    axis.tick_params(axis="y", labelleft=True, labelright=True, labelsize=9)


names_by_group = {group: members.sort_values("date").index for group, members in per_session.groupby("group")}
figure, axes = plt.subplots(len(names_by_group), 1, sharex=True, figsize=(13, 0.22 * len(per_session) + 5),
                            gridspec_kw={"height_ratios": [len(names) for names in names_by_group.values()]})
for axis, (group, names) in zip(axes, names_by_group.items()):
    draw_rows(axis, names, per_session)
    axis.set_title(f"{group}: {len(names)} sessions", loc="left", fontsize=11)
    axis.xaxis.set_major_locator(MultipleLocator(10))
    axis.tick_params(axis="x", labelbottom=True, labelsize=9)            # minutes under every panel, not only the last
    axis.set_xlabel("minutes on the recording clock (pauses included)", fontsize=9)
    axis.grid(axis="x", color="black", alpha=0.12, linewidth=0.6)
    axis.set_axisbelow(False)
figure.legend(handles=[Patch(facecolor=color, label=name) for name, color in EPOCH_COLORS.items()] + [
    Patch(facecolor=PAUSE_COLOR, label="pause in the clock (minutes written on it)"),
    Patch(facecolor="white", edgecolor=BOX_COLOR, label="recorded, in no epoch"),
    Patch(facecolor=LOST_COLOR, label="both lights lost for over 10 s")], loc="upper center", ncol=3, fontsize=11)
plt.tight_layout(rect=(0, 0, 1, 0.94))
plt.show()

jump_table.round(1)

#### 1.2.3. The two lights, with unfilled samples set to NaN
The lab marks a sample it could not fill with the value 1 for both coordinates, and also sets a flag. We trust the flag, never the pixel value. For every tracking file this cell checks:
- the rows at the sentinel (1, 1) are exactly the rows the flag marks unfilled;
- how much of each light is unfilled, over the whole file and inside the lab's epochs (a long loss between epochs would inflate the whole-file figure);
- how many filled positions fall outside the 640 by 480 camera frame (the lab's gap filling can overshoot it).

The tables then rank the sessions with the most unfilled front or back light. The heatmaps show where the animal was in each epoch.


In [ ]:
FRAME_WIDTH, FRAME_HEIGHT = 640, 480           # the camera frame in pixels


def lights(proc, unfilled):
    """The front and back light positions in camera pixels, NaN wherever that light's unfilled flag is set."""
    front, back = proc[:, 2:4].astype(float), proc[:, 4:6].astype(float)   # astype makes copies
    front[unfilled[:, 0]] = np.nan
    back[unfilled[:, 1]] = np.nan
    return front, back


def read_epoch_spans(session):
    """The lab's (first frame, last frame) pair for each epoch, one row per epoch; no rows when the file has no timeStamps."""
    stamps = load_mat(session["analysis"], variables=["timeStamps"]).get("timeStamps", [])
    return np.atleast_1d(stamps).astype(int).reshape(-1, 2)


def describe_lights(session):
    """One tracking file's lights: unfilled fractions for the whole file and inside the epochs, and two checks."""
    tracking = load_mat(session["dvt"], variables=["processedDVT", "samplesUnfilled"])
    processed, unfilled = tracking["processedDVT"], tracking["samplesUnfilled"].astype(bool)
    in_epoch = np.zeros(len(processed), dtype=bool)
    for first_frame, last_frame in read_epoch_spans(session):
        in_epoch[first_frame - 1:last_frame] = True
    front, back = lights(processed, unfilled)
    at_sentinel_front = (processed[:, 2] == 1) & (processed[:, 3] == 1)
    at_sentinel_back = (processed[:, 4] == 1) & (processed[:, 5] == 1)
    positions = np.vstack([front, back])                              # NaN never compares as outside
    outside_frame = ((positions[:, 0] < 1) | (positions[:, 0] > FRAME_WIDTH)
                     | (positions[:, 1] < 1) | (positions[:, 1] > FRAME_HEIGHT))
    return {
        "group": session["group"],
        "share_in_epochs": in_epoch.mean(),
        "unfilled_front": unfilled[:, 0].mean(),
        "unfilled_back": unfilled[:, 1].mean(),
        "unfilled_front_in_epochs": unfilled[in_epoch, 0].mean() if in_epoch.any() else np.nan,
        "unfilled_back_in_epochs": unfilled[in_epoch, 1].mean() if in_epoch.any() else np.nan,
        "sentinel_equals_flags": bool(np.array_equal(at_sentinel_front, unfilled[:, 0])
                                      and np.array_equal(at_sentinel_back, unfilled[:, 1])),
        "samples_outside_frame": int(outside_frame.sum()),
    }


lights_table = pd.DataFrame.from_dict({name: describe_lights(session) for name, session in scope.items()}, orient="index")

lights_by_group = lights_table.groupby("group").agg(
    sessions=("unfilled_front", "size"),
    median_share_in_epochs=("share_in_epochs", "median"),
    front_whole_file=("unfilled_front", "median"), front_in_epochs=("unfilled_front_in_epochs", "median"),
    back_whole_file=("unfilled_back", "median"), back_in_epochs=("unfilled_back_in_epochs", "median"),
    samples_outside_frame=("samples_outside_frame", "sum"),
).round(3)
print(lights_by_group.to_string())
print(f"\nsentinel rows equal the unfilled flags in every file: {bool(lights_table['sentinel_equals_flags'].all())}")
print(f"samples outside the camera frame: {lights_table['samples_outside_frame'].sum()} "
      f"in {(lights_table['samples_outside_frame'] > 0).sum()} sessions")
for light in ("front", "back"):
    print(f"\nmost unfilled {light} light (whole file, then inside the epochs):")
    print(lights_table.nlargest(5, f"unfilled_{light}")[
        ["unfilled_front", "unfilled_back", f"unfilled_{light}_in_epochs", "share_in_epochs"]].round(3).to_string())


In [ ]:
HEATMAP_BINS = (64, 48)                         # cells of 10 by 10 pixels
CELL_PIXELS = FRAME_WIDTH // HEATMAP_BINS[0]
MIN_FRAMES_IN_CELL = 30                         # half a second, for the extent table
ROW_NAMES = ["arena", "track", "variant, 45 degrees", "variant, 90 degrees", "variant, same maze"]


def variant_kind(pattern):
    """The kind of a session's variant segment, from the tags in its analysis file name."""
    tags = [token.lower() for token in pattern.split("_") if token.lower() not in ("tt", "rmaps")]
    if any(tag.startswith("plus45") for tag in tags):
        return "variant, 45 degrees"
    if any(tag.startswith("plusquarter") for tag in tags):
        return "variant, 90 degrees"
    return "variant, same maze"                  # infusion or dark: the track again


def epoch_occupancy(session):
    """Frames the midpoint of the two lights spent in each 10 by 10 pixel cell, for each of the lab's epochs."""
    tracking = load_mat(session["dvt"], variables=["processedDVT", "samplesUnfilled"])
    front, back = lights(tracking["processedDVT"], tracking["samplesUnfilled"].astype(bool))
    midpoint = (front + back) / 2                # NaN wherever either light is unfilled
    row_names = ["arena", "track", variant_kind(session["pattern"])]
    occupancy = {}
    for row_name, (first_frame, last_frame) in zip(row_names, read_epoch_spans(session)):
        visited = midpoint[first_frame - 1:last_frame]
        visited = visited[~np.isnan(visited).any(axis=1)]
        occupancy[row_name], _, _ = np.histogram2d(visited[:, 0], visited[:, 1], bins=HEATMAP_BINS,
                                                   range=[[0, FRAME_WIDTH], [0, FRAME_HEIGHT]])
    return occupancy


def visited_extent(counts):
    """The x and y pixel range of the cells with at least MIN_FRAMES_IN_CELL frames."""
    busy = counts >= MIN_FRAMES_IN_CELL
    busy_x, busy_y = np.flatnonzero(busy.any(axis=1)), np.flatnonzero(busy.any(axis=0))
    return (busy_x[0] * CELL_PIXELS, (busy_x[-1] + 1) * CELL_PIXELS,
            busy_y[0] * CELL_PIXELS, (busy_y[-1] + 1) * CELL_PIXELS)


occupancy_by_map, sessions_by_map = {}, {}
for session in scope.values():
    for row_name, counts in epoch_occupancy(session).items():
        key = (session["group"], row_name)
        occupancy_by_map[key] = occupancy_by_map.get(key, 0) + counts
        sessions_by_map[key] = sessions_by_map.get(key, 0) + 1

extents = pd.DataFrame(
    [(group, row_name, sessions_by_map[(group, row_name)], *visited_extent(counts))
     for (group, row_name), counts in sorted(occupancy_by_map.items())],
    columns=["group", "map", "sessions", "x_min", "x_max", "y_min", "y_max"])
print(f"visited extent in camera pixels (the frame is 0 to {FRAME_WIDTH} by 0 to {FRAME_HEIGHT}):")
print(extents.to_string(index=False))

groups = sorted({group for group, _ in occupancy_by_map})
figure, axes = plt.subplots(len(ROW_NAMES), len(groups), figsize=(4 * len(groups), 3.0 * len(ROW_NAMES)),
                            sharex=True, sharey=True, squeeze=False)
for row, row_name in enumerate(ROW_NAMES):
    for column, group in enumerate(groups):
        axis = axes[row, column]
        axis.set_facecolor("black")
        counts = occupancy_by_map.get((group, row_name))
        if counts is None:
            axis.text(0.5, 0.5, "no sessions", transform=axis.transAxes, ha="center", va="center", color="0.6")
        else:
            axis.imshow(np.log10(1 + counts.T), extent=(0, FRAME_WIDTH, FRAME_HEIGHT, 0), cmap="magma", aspect="equal")
            axis.text(0.02, 0.97, f"{sessions_by_map[(group, row_name)]} sessions", transform=axis.transAxes,
                      va="top", fontsize=8, color="white")
        axis.set_xlim(0, FRAME_WIDTH)
        axis.set_ylim(FRAME_HEIGHT, 0)                          # camera y points down
        if row == 0:
            axis.set_title(group, fontsize=12)
        if column == 0:
            axis.set_ylabel(row_name, fontsize=11)
figure.suptitle("where the midpoint of the two lights was in the camera frame, pooled over sessions "
                "(each map on its own log scale)", fontsize=11)
plt.tight_layout()
plt.show()


#### 1.2.4. Is the light bar rigid?

The two lights sit on a fixed bar on the rat's head, so their distance should barely change, and the heading (the direction from the back light to the front light) should change only a little from one frame to the next. Two views, over all 109 files:
- **Distance between the lights**, one dot per session (the median over its frames) with a bar from the 5th to the 95th percentile, sessions in date order. A rigid bar gives a flat row of dots with short bars.
- **Heading change between consecutive frames**: how often the heading jumps by a large angle in 1/60 s. A head cannot turn 90 degrees in 1/60 s (5,400 degrees per second), so large changes must come from the measurement. The second figure separates the frame pairs where the lights are at least 8 px apart in both frames, because a pixel or two of jitter swings the heading a lot when the lights are close.


In [ ]:
STEP_BINS = np.arange(0, 181, 1.0)               # one-degree bins for the heading change between consecutive frames
MIN_SEPARATION_PX = 8                            # with the lights closer than this, a pixel or two of jitter swings the heading
SCALE_TOP_PX = 40                                # top of the distance panels; longer whiskers get a triangle and a number
OFF_SCALE_COLOR = "#d47264"


def describe_light_bar(session):
    """One tracking file's light bar: the distance between the two lights and the frame-to-frame heading change."""
    tracking = load_mat(session["dvt"], variables=["processedDVT", "samplesUnfilled", "HDRadians"])
    front, back = lights(tracking["processedDVT"], tracking["samplesUnfilled"].astype(bool))
    separation = np.hypot(front[:, 0] - back[:, 0], front[:, 1] - back[:, 1])      # pixels, NaN where a light is unfilled
    heading_step = np.degrees(np.abs(np.angle(np.exp(1j * np.diff(tracking["HDRadians"].astype(float))))))
    has_step = ~np.isnan(heading_step)
    well_separated = has_step & (separation[:-1] >= MIN_SEPARATION_PX) & (separation[1:] >= MIN_SEPARATION_PX)
    return {
        "group": session["group"],
        "separation_median": np.nanmedian(separation),
        "separation_low": np.nanpercentile(separation, 5),
        "separation_high": np.nanpercentile(separation, 95),
        "step_counts": np.histogram(heading_step[has_step], bins=STEP_BINS)[0],
        "step_counts_separated": np.histogram(heading_step[well_separated], bins=STEP_BINS)[0],
    }


bar_table = pd.DataFrame.from_dict({name: describe_light_bar(session) for name, session in scope.items()}, orient="index")
bar_table["date"] = bar_table.index.str.split("_").str[2]
groups = sorted(bar_table["group"].unique())
typical = bar_table["separation_median"].median()

print(f"distance between the two lights: session medians from {bar_table['separation_median'].min():.1f} to "
      f"{bar_table['separation_median'].max():.1f} px, median {typical:.1f} px")
odd = bar_table[(bar_table["separation_high"] > 2 * typical) | (bar_table["separation_low"] < typical / 2)
                | (bar_table["separation_median"] < 0.75 * typical)]
print("\nsessions with an unusual light distance (95th percentile over twice the typical, 5th under half, or median under 3/4):")
print(odd[["group", "separation_low", "separation_median", "separation_high"]].round(1).to_string())

figure, axes = plt.subplots(1, len(groups), figsize=(14, 4.2), sharey=True)
for axis, group in zip(axes, groups):
    members = bar_table[bar_table["group"] == group].sort_values("date")
    positions = np.arange(len(members))
    axis.vlines(positions, members["separation_low"], members["separation_high"].clip(upper=SCALE_TOP_PX),
                color="0.7", linewidth=1.5)
    axis.scatter(positions, members["separation_median"], color="#59a89c", s=22, zorder=3)
    off_scale = (members["separation_high"] > SCALE_TOP_PX).to_numpy()
    axis.scatter(positions[off_scale], np.full(off_scale.sum(), SCALE_TOP_PX), marker="^", color=OFF_SCALE_COLOR,
                 s=40, zorder=4, clip_on=False)
    for position, high in zip(positions[off_scale], members["separation_high"][off_scale]):
        axis.annotate(f"{high:.0f}", (position, SCALE_TOP_PX), textcoords="offset points", xytext=(0, -14),
                      ha="center", fontsize=9, color=OFF_SCALE_COLOR)
    axis.set_ylim(0, SCALE_TOP_PX)
    axis.set_title(f"{group}: {len(members)} sessions, in date order", fontsize=11)
    axis.set_xticks([])
axes[0].set_ylabel("distance between the lights (px)", fontsize=11, labelpad=8)
figure.suptitle("dot: session median; bar: 5th to 95th percentile; red triangle: a bar that runs off the top "
                "(its 95th percentile in px)", fontsize=10)
plt.tight_layout(rect=(0, 0, 1, 0.94))
plt.show()


In [ ]:
from matplotlib.ticker import NullFormatter, ScalarFormatter

ALL_COLOR, SEPARATED_COLOR, REFERENCE_COLOR = "#d55e00", "#0072b2", "0.35"

all_counts = np.sum(bar_table["step_counts"].to_list(), axis=0)
separated_counts = np.sum(bar_table["step_counts_separated"].to_list(), axis=0)
degrees = STEP_BINS[1:-1]                        # the log axis starts at 1 degree, so the first bin is not drawn
total_pairs = all_counts.sum()

figure, axis = plt.subplots(figsize=(10, 5.4))
axis.axvspan(90, 180, color="0.93", zorder=0)
axis.text(95, .8, "Over 90 deg.", fontsize=11, color=REFERENCE_COLOR)
for name, counts, color in (("all frame pairs", all_counts, ALL_COLOR),
                            (f"only pairs with the lights at least {MIN_SEPARATION_PX} px apart", separated_counts, SEPARATED_COLOR)):
    share_at_least = counts[::-1].cumsum()[::-1] / counts.sum()            # share of pairs in this bin or any larger
    axis.plot(degrees, np.where(share_at_least[1:] > 0, share_at_least[1:], np.nan), color=color, linewidth=2.5,
              label=f"{name}\n{counts.sum():,} pairs, {counts[90:].sum():,} Over 90 Deg")
for pairs in (100, 10_000):
    axis.axhline(pairs / total_pairs, color=REFERENCE_COLOR, linestyle="--", linewidth=1)
    axis.text(1.05, pairs / total_pairs * 1.15, f"{pairs:,} pairs", fontsize=11, color=REFERENCE_COLOR)
axis.set_xscale("log")
axis.set_yscale("log")
axis.set_xlim(1, 180)
axis.set_ylim(40 / total_pairs, 1.5)
axis.set_xticks([1, 2, 5, 10, 20, 45, 90, 180])
axis.xaxis.set_major_formatter(ScalarFormatter())
axis.xaxis.set_minor_formatter(NullFormatter())
axis.tick_params(labelsize=11)
axis.set_xlabel("heading change between consecutive frames (degrees)", fontsize=12)
axis.set_ylabel("share of frame pairs with at least this change", fontsize=12)
axis.set_title(f"{total_pairs:,} frame pairs from {len(bar_table)} sessions", fontsize=12)
axis.legend(fontsize=11, loc="upper right", framealpha=0.95)
plt.tight_layout()
plt.show()


Each line is the share of frame pairs whose heading changed by at least the angle on the horizontal axis. Both axes are logarithmic, because the interesting part is how few pairs are in the tail. The dashed lines mark 100 and 10,000 pairs. The gap between the two lines at large angles comes from frames where the lights are close together, where pixel jitter, not a head turn, produces the jump. The legend gives the exact number of pairs over 90 degrees for each line.


### 1.3 · The heading offset: how the light bar sits on the head

#### Why this was examined
`HDRadians` is the direction the light bar points, and the bar is fixed to the head at some angle that need not match the direction the nose points. While the rat runs it mostly moves where its nose points, so the heading minus the direction of motion, averaged over running frames, measures that mounting angle. Stage 2 computes it per recording, as `hd_offset_rad`.

#### The question and the goal
Is there one stable offset per session, and can it be computed from the tracking alone, without path or run labels, so that it exists in the arena too? Which choices (the point whose motion is used, the step length, the speed floor) make it well defined? Goal: a definition stage 2 can compute for every session.

#### What we found
Looked at here in one session, JL1_rec23:
- The offset is about 105 degrees: the bar sits roughly sideways on the head.
- It barely depends on which point's motion is used (midpoint 105.5, front light 104.4, back light 108.2 degrees).
- It stays between 103 and 109 degrees whatever the speed floor and whichever segment is used. The spread of the difference shrinks as the floor rises, and the arena is looser than the track.
- The bar is only about 18 px long, so a pixel of jitter in one light is a few degrees of heading. That sets a floor on how tight the difference can get.

#### What it changed
- `hd_offset_rad` is the circular mean of heading minus direction of motion over running frames: the direction of motion comes from the midpoint of the two lights, over a 6-frame step (0.1 s), and running means faster than 30 px/s (raised from 10).
- Removing the offset makes the average difference zero by construction. The offset column therefore cannot be used to test whether the head follows travel. That is what 1.4 does, from the spread around the offset.
- The offset is an analysis column, not an RNN input.

#### Where it lives
Dictionary: 1.4 and 3.1. The converter does not compute it.


#### 1.3.1. Load one session and define the two measures
One session, JL1_rec23. For each frame we need the direction the rat is moving and the direction the bar points.
- **Direction of motion**: from the position 3 frames before to the position 3 frames after (a 6-frame step, 0.1 s), assigned to the middle frame. A shorter step lets a pixel of jitter dominate, and a longer one averages a turn away.
- **Speed**: the length of that step divided by 0.1 s, in pixels per second.
- **Heading minus direction of motion**, wrapped into ±180 degrees. This is the offset we are after.
- `circular_mean` averages angles as arrows of length 1 and returns the mean direction and R, the length of the mean arrow. R is 1 when the angles are all equal and near 0 when they are spread around the circle.

Camera coordinates: y points down, so angles grow clockwise on the screen.


In [ ]:
STEP = 6  # frames between the two samples that make a direction of motion

sid = find(sessions, "JL1_rec23")
tracking = load_mat(sessions[sid]["dvt"], variables=["processedDVT", "samplesUnfilled", "HDRadians"])
front, back = lights(tracking["processedDVT"], tracking["samplesUnfilled"].astype(bool))
heading = tracking["HDRadians"].astype(float)[STEP // 2:-(STEP // 2)]    # frame i + 3 is the middle of samples i and i + 6


def motion(point):
    """Speed in px/s and heading minus direction of motion in radians, at the middle frame of each 6-frame step."""
    move = point[STEP:] - point[:-STEP]
    speed = np.hypot(move[:, 0], move[:, 1]) / (STEP * DT)
    return speed, np.angle(np.exp(1j * (heading - np.arctan2(move[:, 1], move[:, 0]))))   # camera coordinates, y down


def circular_mean(angle):
    """Mean direction and mean resultant length R of a set of angles."""
    mean_arrow = np.exp(1j * angle).mean()
    return np.angle(mean_arrow), np.abs(mean_arrow)


#### 1.3.2. Does it matter which point's motion we use?
The direction of motion can come from the midpoint of the two lights, the front light or the back light. All three are tried on the frames faster than 10 px/s.

For JL1_rec23 they give 105.5, 104.4 and 108.2 degrees, each with R about 0.5, so the choice moves the offset by only a few degrees. We use the midpoint: it is the centre of the bar and favours neither light.


In [ ]:
print(sid)
for name, point in (("midpoint of the two lights", (front + back) / 2), ("front light", front), ("back light", back)):
    speed, difference = motion(point)
    running = (speed > 10) & ~np.isnan(difference)          # a first look: any frame faster than 10 px/s is running
    mean, length = circular_mean(difference[running])
    print(f"  {name:27s} {np.degrees(mean):7.1f} deg   R {length:.3f}   {int(running.sum())} running frames")


#### 1.3.3. Does the speed floor or the epoch matter?
Frames where the rat is nearly still have a poorly defined direction of motion, so which frames count as running could change the answer. The midpoint's offset is computed for floors of 10, 30, 60 and 100 px/s over the whole session, the arena segment and the track segment. Columns: frames used, offset in degrees, R.

For JL1_rec23:
- The offset stays between 103 and 109 degrees in every row.
- R climbs with the floor: faster frames give a cleaner direction of motion.
- The arena is looser than the track at every floor (R 0.69 against 0.93 at 100 px/s): in the open arena the head turns more independently of travel.

We use 30 px/s. The offset is already stable at 10, so 30 is a compromise, not a natural boundary: it drops the slowest frames and keeps most of the data.


In [ ]:
speed, difference = motion((front + back) / 2)
middle_frame = np.arange(len(speed)) + STEP // 2 + 1                # 1-based frame number of each step's middle
arena_span, track_span = read_epoch_spans(sessions[sid])[:2]
print("midpoint, by epoch and speed floor: frames, offset (deg), R")
for epoch, (first_frame, last_frame) in (("whole", (1, len(front))), ("arena", arena_span), ("track", track_span)):
    inside = (middle_frame >= first_frame) & (middle_frame <= last_frame)
    for floor in (10, 30, 60, 100):
        pick = inside & (speed > floor) & ~np.isnan(difference)
        mean, length = circular_mean(difference[pick])
        print(f"  {epoch:6s} over {floor:3d} px/s: {int(pick.sum()):6d} frames  {np.degrees(mean):7.1f}  R {length:.3f}")


#### 1.3.4. The shape of the distribution
Heading minus direction of motion for every running frame (faster than 30 px/s) of JL1_rec23, before any offset is removed. The dashed line is the circular mean, the offset. Chapter 1.4 asks how wide this peak is and what it depends on.


In [ ]:
speed, difference = motion((front + back) / 2)
running = (speed > 30) & ~np.isnan(difference)
offset_degrees = np.degrees(circular_mean(difference[running])[0])

figure, axis = plt.subplots(figsize=(8, 3.6))
axis.hist(np.degrees(difference[running]), bins=np.arange(-180, 181, 10), color="#59a89c")
axis.axvline(offset_degrees, color="black", linestyle="--")
axis.set_xticks(range(-180, 181, 45))
axis.set_xlabel("heading minus direction of motion (degrees)", fontsize=12)
axis.set_ylabel("frames", fontsize=12)
axis.set_title(f"{sid}: frames over 30 px/s, offset {offset_degrees:.0f}°", fontsize=12)
plt.show()


### 1.4 · Head direction against direction of travel

#### Why this was examined
Synthetic trajectories need a direction of travel, to move the animal, and a head direction. The two are often close and sometimes far apart, for example in a turn. The real recordings say how the head direction is distributed around the direction of travel and what that depends on.

#### The question and the goal
How closely does the direction the head points follow the direction the rat is moving, and does that differ between the open arena and the maze, or from rat to rat? Goal: a distribution of the difference between the two directions, by context and by animal, that a synthetic-trajectory generator can later sample from.

#### What we found
Baseline sessions, running frames (over 30 px/s), each animal pooled over its sessions.
- **The head follows travel about twice as closely on the maze.** On the maze about 70% of frames have the head within 30 degrees of the direction of travel, and the typical difference (circular standard deviation) is 39 to 44 degrees. In the arena only about a third of frames are within 30 degrees, and the typical difference is 59 to 71 degrees. The four animals look alike.
- **In the arena the head is more often clockwise of travel**, for NS15 and NS16 (towards +45 to +90 degrees). This is an observation, not an explanation.
- **The scatter shrinks with speed on the maze but not in the arena.** It falls from about 100 degrees at 30 px/s to about 17 degrees at the highest speeds on the maze, and levels off near 50 degrees in the arena. Measurement noise does not explain it: 1 px of error in each light would scatter the direction by about 19 degrees at 30 px/s and 6 degrees at 100 px/s.
- **While the rat turns, the head direction falls behind the direction of travel.** The slope near zero turning is -41 to -50 ms on the maze and -124 to -179 ms in the arena.
- **That slope may be geometry, and is not yet tested.** The tracked midpoint is ahead of the body's pivot, so it moves in a direction rotated into a turn, which alone gives a negative slope of about minus the distance ahead of the pivot divided by speed.

#### What it changed
Nothing in the converter, which stores the heading as given and the offset defined in 1.3. It matters for stage 3 and the trajectory generator:
- Head direction relative to travel is not one distribution. It depends on context, on speed and, while turning, on the turning rate.
- Head direction stays an analysis column and is not an RNN input unless a control config says so (AGENTS.md rule 1).

#### Where it lives
Dictionary: 3 and 3.1. The stage 3 statistics spec, when it exists.

#### Things to try
- [ ] Fit the turning slope in separate speed bands. If slope times speed is roughly constant, the slope is geometry.
- [ ] Repeat with the front light, then the back light, as the tracked point.
- [ ] Compare the infusion sessions segment by segment, and find out which segments are under the drug.
- [ ] Add the 45 and 90 degree rotation segments.


#### 1.4.1. The difference, frame by frame, pooled by animal and context
Each baseline session gives, for every frame, a speed, a direction of travel, a head direction and a turning rate. The session's offset is subtracted, which leaves head minus travel. The frames are counted into bins for each animal and context, and the counts are added over its sessions (steps 1 to 9 under "How the polar figure was generated" below). The printout is the number of running frames behind each animal and context.


In [ ]:
from collections import Counter

RUNNING_FLOOR = 30                               # px/s: faster frames count as running, as the converter will do it
MANIPULATION_TAGS = {"bilatmusc", "leftmusc", "bilatacsf", "leftacsf", "dark"}
CONTEXTS = ("arena", "maze")                     # the lab's segment 1 and segment 2
RESIDUAL_BINS = np.arange(-180, 181, 5)          # degrees
RESIDUAL_CENTERS = RESIDUAL_BINS[:-1] + 2.5
SPEED_EDGES = np.array([5, 10, 20, 30, 45, 60, 80, 100, 150, 300.0])     # px/s
TURN_EDGES = np.arange(-400, 401, 40.0)                                  # degrees per second


def wrap(angle):
    """Angles wrapped into the range minus pi to pi."""
    return np.angle(np.exp(1j * angle))


def circular_stats_by_bin(values, bin_edges, angles):
    """Per bin of values: the number of frames and the sums of cos and sin of the angles, which add across sessions."""
    size = len(bin_edges) - 1
    index = np.digitize(values, bin_edges) - 1
    inside = (index >= 0) & (index < size)
    return np.array([np.bincount(index[inside], minlength=size),
                     np.bincount(index[inside], weights=np.cos(angles[inside]), minlength=size),
                     np.bincount(index[inside], weights=np.sin(angles[inside]), minlength=size)])


def is_manipulated(pattern):
    """True for the infusion and dark sessions, whose whole file is under a manipulation."""
    return bool({token.lower() for token in pattern.split("_")} & MANIPULATION_TAGS)


def describe_travel_and_head(session):
    """Head direction minus direction of travel in the arena and the maze, or None without both segments."""
    epoch_spans = read_epoch_spans(session)
    if len(epoch_spans) < 2:
        return None
    tracking = load_mat(session["dvt"], variables=["processedDVT", "samplesUnfilled", "HDRadians"])
    front, back = lights(tracking["processedDVT"], tracking["samplesUnfilled"].astype(bool))
    midpoint = (front + back) / 2
    move = midpoint[STEP:] - midpoint[:-STEP]
    speed = np.hypot(move[:, 0], move[:, 1]) / (STEP * DT)                    # px/s, at the middle frame of each step
    travel = np.arctan2(move[:, 1], move[:, 0])                               # camera coordinates, y down
    heading = tracking["HDRadians"].astype(float)[STEP // 2:-(STEP // 2)]
    difference = wrap(heading - travel)
    running = (speed > RUNNING_FLOOR) & ~np.isnan(difference)
    offset = np.angle(np.exp(1j * difference[running]).mean())                # the session's hd_offset_rad
    residual = wrap(difference - offset)                                       # head direction minus travel direction
    turning = np.full(len(travel), np.nan)                                     # degrees per second, change of travel direction
    turning[STEP // 2:-(STEP // 2)] = np.degrees(wrap(travel[STEP:] - travel[:-STEP])) / (STEP * DT)
    middle_frame = np.arange(len(travel)) + STEP // 2 + 1                       # 1-based frame number of each step's middle

    summary = {}
    for context, (first_frame, last_frame) in zip(CONTEXTS, epoch_spans[:2]):
        in_context = (middle_frame >= first_frame) & (middle_frame <= last_frame) & ~np.isnan(difference)
        running_here = in_context & running
        turning_here = running_here & ~np.isnan(turning)
        summary[context] = {
            "frames": int(running_here.sum()),
            "residual_counts": np.histogram(np.degrees(residual[running_here]), bins=RESIDUAL_BINS)[0],
            "by_speed": circular_stats_by_bin(speed[in_context], SPEED_EDGES, residual[in_context]),   # all speeds, to show the floor
            "by_turn": circular_stats_by_bin(turning[turning_here], TURN_EDGES, residual[turning_here]),
        }
    return summary


pooled, used, skipped = {}, 0, Counter()
for session in scope.values():
    if is_manipulated(session["pattern"]):
        skipped["infusion or dark"] += 1
        continue
    summary = describe_travel_and_head(session)
    if summary is None:
        skipped["no timeStamps"] += 1
        continue
    used += 1
    for context, statistics in summary.items():
        entry = pooled.setdefault((session["group"], context), {name: 0 for name in statistics})
        for name, value in statistics.items():
            entry[name] = entry[name] + value

groups = sorted({group for group, _ in pooled})
print(f"{used} baseline sessions used; skipped: {dict(skipped)}")
for (group, context), entry in sorted(pooled.items()):
    print(f"  {group:12s} {context:6s} {entry['frames']:9,d} running frames (over {RUNNING_FLOOR} px/s)")


#### 1.4.2. The distribution of the difference, by animal and context


In [ ]:
RADIAL_POWER = 0.5                               # 0.5: bar area is proportional to share; 1.0: bar length is
RING_SHARES = [0.01, 0.03, 0.06, 0.1]            # share of frames per 5 degree bin, drawn as rings
GROUP_COLORS = {"JL1HPCData": "#a559aa", "NS14Data": "#59a89c", "NS15Data": "#082a54", "NS16Data": "#e69f00"}


def group_label(group):
    """The folder name without its trailing Data."""
    return group.removesuffix("Data")


def spread_from_counts(counts):
    """The circular standard deviation, in degrees, of a histogram of angles."""
    angles = np.radians(RESIDUAL_CENTERS)
    resultant = np.hypot((counts * np.cos(angles)).sum(), (counts * np.sin(angles)).sum()) / counts.sum()
    return np.degrees(np.sqrt(-2 * np.log(resultant)))


def share_within(counts, limit_degrees):
    """The share of frames whose difference is within plus or minus limit_degrees."""
    return counts[np.abs(RESIDUAL_CENTERS) < limit_degrees].sum() / counts.sum()


largest_share = max(entry["residual_counts"].max() / entry["residual_counts"].sum() for entry in pooled.values())
radial_max = largest_share ** RADIAL_POWER * 1.1
rings = [share for share in RING_SHARES if share < largest_share]

figure, axes = plt.subplots(len(CONTEXTS), len(groups), subplot_kw={"projection": "polar"},
                            figsize=(4.6 * len(groups), 5.4 * len(CONTEXTS)))
figure.subplots_adjust(left=0.07, right=0.99, top=0.85, bottom=0.11, hspace=0.55, wspace=0.12)
for row, context in enumerate(CONTEXTS):
    for column, group in enumerate(groups):
        axis = axes[row, column]
        counts = pooled[(group, context)]["residual_counts"]
        spread = spread_from_counts(counts)
        axis.set_theta_zero_location("N")
        axis.set_theta_direction(-1)                                   # clockwise, as in the camera image
        wedge = np.linspace(-min(np.radians(spread), np.pi), min(np.radians(spread), np.pi), 100)
        axis.fill_between(wedge, 0, radial_max, color="0.88", zorder=0)           # plus and minus one standard deviation
        for edge in (wedge[0], wedge[-1]):
            axis.plot([edge, edge], [0, radial_max], color="black", linestyle="--", linewidth=1.2)
        axis.bar(np.radians(RESIDUAL_CENTERS), (counts / counts.sum()) ** RADIAL_POWER, width=np.radians(5),
                 color=GROUP_COLORS[group], edgecolor="white", linewidth=0.3)
        axis.set_ylim(0, radial_max)
        axis.set_rgrids([share ** RADIAL_POWER for share in rings], labels=[f"{share:.0%}" for share in rings],
                        angle=157.5, fontsize=12)
        for label in axis.get_yticklabels():
            label.set_bbox({"facecolor": "white", "edgecolor": "none", "alpha": 0.9, "pad": 1.5})
        axis.set_thetagrids([0, 45, 90, 135, 180, 225, 270, 315],
                            labels=["0", "+45", "+90", "+135", "180", "-135", "-90", "-45"], fontsize=13)
        axis.text(0.5, -0.2, f"{counts.sum():,} frames\n{share_within(counts, 30):.0%} within ±30° of travel\n"
                             f"typical difference ±{spread:.0f}°",
                  transform=axis.transAxes, ha="center", va="top", fontsize=13)
        if row == 0:
            axis.set_title(group_label(group), fontsize=20, pad=24)
figure.suptitle("Head direction minus direction of travel", fontsize=20, y=0.985)
figure.text(0.5, 0.925, "Running frames (over 30 px/s) in baseline sessions only (no muscimol, ACSF or dark). "
                        "Head direction is the LED-bar heading minus each session's mounting offset.", ha="center", fontsize=13)
figure.text(0.5, -0.1, "0: the head points the way the rat is moving.  [+°]: head clockwise of travel as seen in the camera "
                        "image, [-°]: counter-clockwise.  180°: head points backwards.\nBars: share of frames per 5° bin "
                        "(rings, square-root scale).  Shaded wedge between dashed lines: ± one circular standard "
                        "deviation, the typical size of the difference.", ha="center", fontsize=13)
for row, context in enumerate(CONTEXTS):                                # row labels, placed once the layout is final
    position = axes[row, 0].get_position()
    figure.text(0.015, (position.y0 + position.y1) / 2, context, rotation=90, ha="center", va="center",
                fontsize=22, fontweight="bold")
plt.show()


#### How the polar figure was generated

1. **Which sessions.** Baseline sessions only. The infusion and dark sessions are left out, because their whole file is
   under a manipulation (names carrying `bilatmusc`, `leftmusc`, `bilatACSF`, `leftACSF` or `dark`). Sessions with no
   `timeStamps` are skipped, because without them there are no arena and maze segments. The cell prints how many
   sessions are used. Each folder is one subject tag (JL1, NS14, NS15, NS16), so each column is one animal.
2. **Where the rat is.** In every frame the tracker gives two lights. A light the lab could not fill is NaN. The
   midpoint of the two lights stands for the head's position.
3. **Direction of travel.** The displacement of the midpoint over 6 frames (0.1 s), as an angle in camera coordinates
   (`atan2(dy, dx)`, y pointing down). The same displacement over 0.1 s gives the speed in px/s. Both are assigned to
   the middle frame of the 6-frame step.
4. **Head direction.** The stored heading, the angle from the back light to the front light, taken at the same middle
   frame. It is the direction the LED bar points, which is not the direction the head faces.
5. **The difference.** Heading minus direction of travel, wrapped into minus 180 to plus 180 degrees, so that 179 and
   -179 degrees are 2 degrees apart.
6. **Running frames.** Frames with a speed over 30 px/s and both angles defined. Slower frames are left out, because
   the direction of travel is unreliable there: at 30 px/s the midpoint moves 3 px in 0.1 s, so an error of one pixel
   changes the direction by about 20 degrees.
7. **The offset.** For each session, the circular mean of the difference over its running frames. This is the mounting
   angle of the bar on the head (`hd_offset_rad`), about 105 degrees in the session we looked at. Subtracting it
   gives the plotted quantity, head direction minus direction of travel. The mean of that quantity over a whole
   session is zero by construction, so the figure shows its spread and shape, not where it centres.
8. **Arena and maze.** The lab's `timeStamps` give the segments: the first is the arena, the second the maze (the
   track). A step belongs to a context when its middle frame lies inside that segment. The third segment, the
   variant, is not used here.
9. **Pooling.** For each animal and context, the differences go into 5 degree bins, and the bin counts are added over
   the animal's sessions. Dividing by the total gives the share of frames per bin.

#### Limits
- The standard deviation assumes a bell shape. These distributions have a peak and long tails, so it overstates the
  typical difference, and the within-30-degrees share is the more direct number.
- The direction of travel is that of the midpoint of two lights, not of the body, and position noise of about a pixel
  makes it unreliable at low speed. The heading itself jitters by a few degrees from the same noise on an 18 px bar.
- Only running frames are shown. How head and travel relate while the rat is slow or still is not in this figure.


#### 1.4.3. What the difference depends on: speed and turning


In [ ]:
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import MultipleLocator, NullFormatter, ScalarFormatter

MIN_FRAMES_PER_BIN = 300                         # bins with fewer frames are not drawn
TURNING_FIT_LIMIT = 200                          # degrees per second: the turning slope is fitted inside this range
CONTEXT_STYLE = {"maze": ("#0072b2", "-"), "arena": ("#e69f00", "--")}
speed_centers = np.sqrt(SPEED_EDGES[:-1] * SPEED_EDGES[1:])
turn_centers = 0.5 * (TURN_EDGES[:-1] + TURN_EDGES[1:])


def circular_summary(statistics):
    """Per bin: the frame count, the circular mean and the circular standard deviation in degrees."""
    count, cosine_sum, sine_sum = statistics
    with np.errstate(invalid="ignore", divide="ignore"):
        mean = np.degrees(np.arctan2(sine_sum, cosine_sum))
        spread = np.degrees(np.sqrt(-2 * np.log(np.hypot(cosine_sum, sine_sum) / count)))
    return count, mean, spread


def turning_slope(centers, mean_residual, count):
    """Slope (seconds) and intercept (degrees) of mean head-minus-travel against turning rate, fitted near zero turning."""
    selected = (count >= MIN_FRAMES_PER_BIN) & (np.abs(centers) <= TURNING_FIT_LIMIT)
    return np.polyfit(centers[selected], mean_residual[selected], 1, w=np.sqrt(count[selected]))


def pooled_and_per_rat(context, statistic):
    """The summary pooled over all rats, and the summary of each rat alone, for one context."""
    pooled_over_rats = circular_summary(np.sum([pooled[(group, context)][statistic] for group in groups], axis=0))
    per_rat = [circular_summary(pooled[(group, context)][statistic]) for group in groups]
    return pooled_over_rats, per_rat


def range_over_rats(per_rat, position):
    """Lowest and highest value per bin among single rats (position 1: mean, 2: spread), where a rat has enough frames."""
    values = np.array([np.where(summary[0] >= MIN_FRAMES_PER_BIN, summary[position], np.nan) for summary in per_rat])
    has_data = ~np.isnan(values).all(axis=0)
    low = np.full(values.shape[1], np.nan)
    high = np.full(values.shape[1], np.nan)
    low[has_data] = np.nanmin(values[:, has_data], axis=0)
    high[has_data] = np.nanmax(values[:, has_data], axis=0)
    return low, high


def write_heading(grid_cell, heading, description):
    """A heading and a plain-language description, in an empty strip above a panel."""
    strip = figure.add_subplot(grid_cell)
    strip.axis("off")
    strip.text(0, 1.0, heading, fontsize=16, fontweight="bold", va="top", transform=strip.transAxes)
    strip.text(0, 0.5, description, fontsize=12, va="top", transform=strip.transAxes)


figure = plt.figure(figsize=(15.5, 7.4))
grid = figure.add_gridspec(2, 2, height_ratios=[1.1, 4], hspace=0.3, wspace=0.14, left=0.07, right=0.98, top=0.93, bottom=0.1)
write_heading(grid[0, 0], "A  Head versus travel, by running speed",
              "The scatter of head direction around the direction of travel (circular\nstandard deviation, degrees) for frames in each "
              "speed range. Lower means\nthe head follows travel more closely.")
write_heading(grid[0, 1], "B  Head versus travel, while turning",
              "The average of head direction minus direction of travel at each turning rate\n(+ is clockwise in the camera image). "
              "A falling line: the head falls behind the\ndirection of travel as the rat turns. A rising line: it is ahead.")
speed_axis = figure.add_subplot(grid[1, 0])
turn_axis = figure.add_subplot(grid[1, 1])
slope_lines = []
for context, (color, linestyle) in CONTEXT_STYLE.items():
    (count, _, spread), per_rat = pooled_and_per_rat(context, "by_speed")
    low, high = range_over_rats(per_rat, 2)
    speed_axis.fill_between(speed_centers, low, high, color=color, alpha=0.2)
    enough = count >= MIN_FRAMES_PER_BIN
    speed_axis.plot(speed_centers[enough], spread[enough], linestyle, marker="o", markersize=6, linewidth=2.5, color=color)

    (count, mean, _), per_rat = pooled_and_per_rat(context, "by_turn")
    low, high = range_over_rats(per_rat, 1)
    turn_axis.fill_between(turn_centers, low, high, color=color, alpha=0.2)
    enough = count >= MIN_FRAMES_PER_BIN
    turn_axis.plot(turn_centers[enough], mean[enough], linestyle, marker="o", markersize=6, linewidth=2.5, color=color)
    pooled_slope = turning_slope(turn_centers, mean, count)[0] * 1000
    rat_slopes = [turning_slope(turn_centers, summary[1], summary[0])[0] * 1000 for summary in per_rat]
    slope_lines.append(f"{context}: {pooled_slope:.0f} ms  (single rats {min(rat_slopes):.0f} to {max(rat_slopes):.0f})")

speed_axis.axvline(RUNNING_FLOOR, color="0.5", linestyle="-.", linewidth=1)
speed_axis.text(RUNNING_FLOOR * 1.06, 4, "30 px/s: the running floor", rotation=90, fontsize=11, color="0.4", va="bottom")
speed_axis.set_xscale("log")
speed_axis.set_xticks([5, 10, 30, 100, 300])
speed_axis.xaxis.set_major_formatter(ScalarFormatter())
speed_axis.xaxis.set_minor_formatter(NullFormatter())
speed_axis.set_ylim(0, 150)
speed_axis.yaxis.set_major_locator(MultipleLocator(30))
speed_axis.set_xlabel("speed (px/s)", fontsize=13)
speed_axis.set_ylabel("scatter of head minus travel (degrees)", fontsize=13)
speed_axis.legend(handles=[Line2D([], [], color=color, linestyle=linestyle, marker="o", linewidth=2.5, label=context)
                           for context, (color, linestyle) in CONTEXT_STYLE.items()]
                  + [Patch(facecolor="0.5", alpha=0.25, label="lowest to highest single rat")],
                  fontsize=12, loc="upper right")
turn_axis.axhline(0, color="0.6", linewidth=1)
turn_axis.axvline(0, color="0.6", linewidth=1)
turn_axis.set_xticks(range(-400, 401, 200))
turn_axis.yaxis.set_major_locator(MultipleLocator(10))
turn_axis.set_xlabel("turning rate (degrees per second, + clockwise)", fontsize=13)
turn_axis.set_ylabel("average head minus travel (degrees)", fontsize=13)
turn_axis.text(0.03, 0.04, "slope of the line near zero turning\n(negative: the head falls behind the\ndirection of travel as the rat turns)\n"
               + "\n".join(slope_lines), transform=turn_axis.transAxes, va="bottom", fontsize=12)
for axis in (speed_axis, turn_axis):
    axis.tick_params(labelsize=12)
figure.suptitle("Head direction against direction of travel: the effect of running speed (A) and of turning (B)",
                fontsize=17, y=0.99)
plt.show()


#### How figure 2 was generated
It uses the same frames, offsets and head-minus-travel difference as the polar figure (steps 1 to 9 above), plus two more numbers per frame.
1. **Panel A, speed.** Frames of every speed from 5 px/s up are grouped into speed bands, not only the running ones, so the plot shows what the 30 px/s floor cuts off. In each band, the circular standard deviation of head minus travel, √(−2 ln R), where R measures how tightly the angles cluster (1 is identical). The solid line pools the four animals by adding their frames, so animals with more frames weigh more. The shaded band runs from the lowest to the highest single animal. Bands with fewer than 300 frames are not drawn.
2. **Panel B, turning.** Turning rate is the change in the direction of travel over the same 6-frame step, in degrees per second (+ clockwise in the camera image). Running frames go into 40°/s bins. Each point is the circular mean of head minus travel in that bin, pooled and ranged over animals the same way.
3. **The slope in the text.** A straight line is fitted to the binned means within ±200°/s, each bin weighted by the square root of its frame count. A slope in degrees per (degree per second) is seconds, so it is multiplied by 1000 to give milliseconds. Negative: the head falls behind the direction of travel as the rat turns. The pooled value is printed first, then the lowest to highest single animal.

#### Limits
- Subtracting each session's offset makes the overall average zero by construction. Both panels show how the difference varies with speed and turning, not whether the head follows travel on average.
- A negative slope can also come from geometry: the tracked midpoint is ahead of the body's pivot. This has not been tested.
